# 02 · Silver layer: validation, deduplication, quarantine

Reads the deliberately dirty CDR file (`cdrs_dirty.csv`, generated by `CDR.py` with `dirty_rate=0.05, seed=42`) and splits it into:

- **`bronze_cdrs`**: raw rows exactly as received (all strings) + ingest metadata
- **`silver_cdrs`**: valid, typed, deduplicated rows
- **`quarantine_cdrs`**: rejected rows with their raw values, a reject reason and a timestamp

A reconciliation check at the end asserts that every bronze row lands in exactly one of silver or quarantine.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType

SOURCE_PATH = "/Volumes/workspace/default/raw/cdrs_dirty.csv"

## 1. Bronze: raw ingest

Every column is read as a **string**. Bronze keeps the data exactly as it arrived, so bad values (e.g. `not_a_timestamp`) are preserved as evidence instead of silently becoming null.

In [ ]:
bronze_schema = StructType([
    StructField("call_id",    StringType(), True),
    StructField("caller",     StringType(), True),
    StructField("callee",     StringType(), True),
    StructField("start_ts",   StringType(), True),
    StructField("duration_s", StringType(), True),
    StructField("status",     StringType(), True),
])

bronze = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .csv(SOURCE_PATH)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

bronze.write.mode("overwrite").saveAsTable("bronze_cdrs")
print(spark.table("bronze_cdrs").count())

## 2. Typing, validation and deduplication

- `try_cast` converts strings to typed columns and returns **null** on bad input (a plain `.cast()` throws under ANSI mode). Typed values go into **new** columns so the raw strings are kept.
- Validation rules produce `_error_type`.
- Duplicates are found with `row_number()` over each `call_id`; copy 1 is the original, copies 2+ are duplicates.
- `coalesce` gives **validation errors priority** over duplicate detection: a row is bad regardless of how many copies exist.

In [ ]:
w = Window.partitionBy("call_id").orderBy("_ingested_at")

silver = (
    bronze
    .withColumn("duration_int",    F.expr("try_cast(duration_s AS INT)"))
    .withColumn("start_ts_parsed", F.expr("try_cast(start_ts AS TIMESTAMP)"))
    .withColumn(
        "_error_type",
        F.when(
            F.col("caller").isNull() |
            (F.trim(F.col("caller")) == "") |
            (F.upper(F.trim(F.col("caller"))) == "NULL"),
            "null_caller",
        )
        .when(
            F.col("start_ts_parsed").isNull(),
            "bad_timestamp",
        )
        .when(
            F.col("duration_int") < 0,
            "negative_duration",
        )
        .when(
            (F.col("status") == "ANSWERED") &
            (F.col("duration_int") == 0),
            "answered_zero",
        )
    )
    .withColumn("_rn", F.row_number().over(w))
    .withColumn(
        "_reject_reason",
        F.coalesce(
            F.col("_error_type"),
            F.when(F.col("_rn") > 1, F.lit("duplicate")),
        ),
    )
)

display(silver.groupBy("_reject_reason").count().orderBy("_reject_reason"))

### Reconciliation: quarantine vs injected counts

| Reason | Injected by generator | Caught in silver |
|---|---|---|
| null_caller | 19 | 20 |
| answered_zero | 14 | 14 |
| duplicate | 12 | 10 |
| negative_duration | 10 | 11 |
| bad_timestamp | 9 | 9 |

Two injected duplicates copied rows that were already corrupted. Because validation errors take priority over duplicate detection, both copies are labelled with the original error, which moves 2 rows from `duplicate` into `null_caller` and `negative_duration`.

Total quarantined rows (64) match total injected issues (64). The query below shows every duplicated `call_id` and how each copy was labelled.

In [ ]:
dup_ids = silver.groupBy("call_id").count().filter("count > 1").select("call_id")

display(
    silver.join(dup_ids, "call_id")
          .select("call_id", "_rn", "caller", "duration_s", "_reject_reason")
          .orderBy("call_id", "_rn")
)

## 3. Write `silver_cdrs`

Valid rows only, typed columns renamed back to their normal names, helper columns dropped.

In [ ]:
silver_clean = (
    silver
    .filter(F.col("_reject_reason").isNull())
    .select(
        "call_id",
        "caller",
        "callee",
        F.col("start_ts_parsed").alias("start_ts"),
        F.col("duration_int").alias("duration_s"),
        "status",
        "_ingested_at",
        "_source_file",
    )
)

silver_clean.write.mode("overwrite").saveAsTable("silver_cdrs")
print(spark.table("silver_cdrs").count())

In [ ]:
spark.table("silver_cdrs").printSchema()

## 4. Write `quarantine_cdrs`

Rejected rows with their **raw** string values, the reject reason, and when they were quarantined.

In [ ]:
quarantine = (
    silver
    .filter(F.col("_reject_reason").isNotNull())
    .select(
        "call_id",
        "caller",
        "callee",
        "start_ts",
        "duration_s",
        "status",
        "_ingested_at",
        "_source_file",
        "_reject_reason",
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

quarantine.write.mode("overwrite").saveAsTable("quarantine_cdrs")
print(spark.table("quarantine_cdrs").count())

In [ ]:
display(spark.table("quarantine_cdrs").filter(F.col("_reject_reason") == "bad_timestamp"))

## 5. Reconciliation check

Every bronze row must land in exactly one of silver or quarantine. If not, the notebook (and any job running it) fails.

In [ ]:
bronze_n     = spark.table("bronze_cdrs").count()
silver_n     = spark.table("silver_cdrs").count()
quarantine_n = spark.table("quarantine_cdrs").count()

print(f"bronze={bronze_n}  silver={silver_n}  quarantine={quarantine_n}")

assert bronze_n == silver_n + quarantine_n, (
    f"Row leak: {bronze_n} in, {silver_n + quarantine_n} out"
)
print("Reconciliation OK")